# Online Retail Transaction Clustering Using Machine Learning Techniques

**Course:** Fundamentals of Data Science  
**Project:** Online Retail Transaction Clustering

## Aim
Explore shopping baskets, summarize transaction and product patterns, visualize baskets using PCA, and compare clustering algorithms.

## Important dataset note
This notebook follows the report's stated format: **each row is one transaction and each non-empty cell contains a product name**. It does not assume customer IDs, invoice IDs, dates, prices, or quantities.

Prepare a CSV file in that format before running the analysis. Example:

| Item 1 | Item 2 | Item 3 |
|---|---|---|
| mineral water | eggs | bread |
| bread | milk | *(blank)* |

If your CSV instead has columns such as `InvoiceNo` and `Description`, it is a different format and needs a different preprocessing step.


## 1. Import libraries

If a library is missing, install it in your notebook environment with `pip install pandas numpy matplotlib scikit-learn`.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the transaction CSV

**Google Colab:** the upload dialog will let you choose the CSV from your device.  
**Local Jupyter:** if Colab is not being used, put the CSV in the notebook's folder and enter its filename below.


In [ ]:
def load_transactions():
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file was uploaded.")
        filename = next(iter(uploaded))
        return pd.read_csv(filename)
    except ImportError:
        filename = input("Enter CSV filename/path (for example, transactions.csv): ").strip()
        if not filename:
            raise ValueError("Please enter a CSV filename.")
        return pd.read_csv(filename)

raw_df = load_transactions()
print("Dataset shape:", raw_df.shape)
display(raw_df.head())

## 3. Inspect and clean the data

Blank cells are expected because some baskets contain fewer products than others. Product names are trimmed and lowercased. Repeated products within the same basket are removed for presence/absence analysis. Duplicate baskets are not deleted because different purchases may contain the same products.


In [ ]:
# Convert each row to a list of cleaned product names.
transactions = []
for _, row in raw_df.iterrows():
    items = []
    for value in row.tolist():
        if pd.isna(value):
            continue
        item = str(value).strip().lower()
        if item and item not in items:
            items.append(item)
    transactions.append(items)

# Remove completely empty rows only.
transactions = [items for items in transactions if items]

if not transactions:
    raise ValueError(
        "No product baskets were found. Check that each row contains product names "
        "and that the CSV is in the basket format described above."
    )

n_transactions = len(transactions)
all_products = sorted({item for basket in transactions for item in basket})
basket_sizes = np.array([len(basket) for basket in transactions])

print(f"Non-empty transactions: {n_transactions}")
print(f"Distinct products: {len(all_products)}")
print("First 5 cleaned baskets:")
for basket in transactions[:5]:
    print(basket)

## 4. Transaction-size statistics

In [ ]:
basket_summary = pd.Series(basket_sizes, name="distinct_products").describe()
display(basket_summary.to_frame())

plt.figure(figsize=(8, 4))
bins = min(30, max(5, int(np.sqrt(len(basket_sizes)))))
plt.hist(basket_sizes, bins=bins, edgecolor="black")
plt.title("Distribution of Transaction Basket Sizes")
plt.xlabel("Number of distinct products in a transaction")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

print(f"Mean basket size: {basket_sizes.mean():.2f}")
print(f"Median basket size: {np.median(basket_sizes):.2f}")

## 5. Product frequency

A product's frequency here means the number of transaction baskets containing it, not the number of units sold or revenue.


In [ ]:
from collections import Counter

product_counts = Counter(item for basket in transactions for item in set(basket))
product_frequency = pd.DataFrame(
    product_counts.items(), columns=["product", "transaction_frequency"]
).sort_values("transaction_frequency", ascending=False)

display(product_frequency.head(15))

top_n = min(10, len(product_frequency))
top_products = product_frequency.head(top_n).sort_values("transaction_frequency")
plt.figure(figsize=(9, 5))
plt.barh(top_products["product"], top_products["transaction_frequency"])
plt.title(f"Top {top_n} Products by Transaction Frequency")
plt.xlabel("Number of transaction baskets containing product")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 6. Build the binary transaction–product matrix

Each row is a transaction and each column is a product. `1` means the product appears in the basket; `0` means it does not.


In [ ]:
product_to_col = {product: i for i, product in enumerate(all_products)}
X = np.zeros((len(transactions), len(all_products)), dtype=np.uint8)

for row_idx, basket in enumerate(transactions):
    for product in set(basket):
        X[row_idx, product_to_col[product]] = 1

basket_matrix = pd.DataFrame(X, columns=all_products)
print("Binary matrix shape:", basket_matrix.shape)
display(basket_matrix.head())

## 7. PCA visualization

PCA reduces the high-dimensional binary matrix to two dimensions for an exploratory plot. The plot is an approximation; apparent separation does not prove that natural clusters exist.


In [ ]:
# PCA requires at least two features and two rows.
if X.shape[0] >= 2 and X.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    print("Variance retained by first two components:",
          f"{pca.explained_variance_ratio_.sum():.4f}")
    print("Component variance ratios:", pca.explained_variance_ratio_)

    # Plot a sample for readability when the dataset is large.
    plot_n = min(5000, len(X_pca))
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(len(X_pca), size=plot_n, replace=False)
    plt.figure(figsize=(8, 5))
    plt.scatter(X_pca[sample_idx, 0], X_pca[sample_idx, 1], s=12, alpha=0.6)
    plt.title("PCA Projection of Transactions")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.tight_layout()
    plt.show()
else:
    X_pca = None
    print("PCA plot skipped: at least 2 transactions and 2 distinct products are needed.")

## 8. Compare clustering algorithms

Clustering algorithms can be computationally expensive for large transaction datasets. To keep this notebook practical, the comparison uses a reproducible sample of up to 2,000 transactions. Scores are only comparable when algorithms use the same rows and feature representation.

- **Silhouette score:** higher is generally better.
- **Davies–Bouldin index:** lower is generally better.
- For DBSCAN, noise points are labelled `-1`; scores are calculated only if at least two non-noise clusters remain.


In [ ]:
MAX_CLUSTER_ROWS = 2000
rng = np.random.default_rng(RANDOM_STATE)
if len(X) > MAX_CLUSTER_ROWS:
    cluster_idx = np.sort(rng.choice(len(X), size=MAX_CLUSTER_ROWS, replace=False))
else:
    cluster_idx = np.arange(len(X))

X_cluster = X[cluster_idx].astype(float)
print("Rows used for clustering:", len(X_cluster), "of", len(X))

def evaluate_labels(X_data, labels, model_name):
    labels = np.asarray(labels)
    # DBSCAN noise is excluded from metrics; other algorithms have no noise label.
    if model_name == "DBSCAN":
        keep = labels != -1
        X_eval, labels_eval = X_data[keep], labels[keep]
    else:
        X_eval, labels_eval = X_data, labels

    unique_labels = np.unique(labels_eval)
    if len(unique_labels) < 2 or len(unique_labels) >= len(labels_eval):
        return {"model": model_name, "clusters_excluding_noise": len(unique_labels),
                "noise_points": int(np.sum(labels == -1)) if model_name == "DBSCAN" else 0,
                "silhouette": np.nan, "davies_bouldin": np.nan}

    try:
        sil = silhouette_score(X_eval, labels_eval)
        dbi = davies_bouldin_score(X_eval, labels_eval)
    except Exception:
        sil, dbi = np.nan, np.nan

    return {"model": model_name, "clusters_excluding_noise": len(unique_labels),
            "noise_points": int(np.sum(labels == -1)) if model_name == "DBSCAN" else 0,
            "silhouette": sil, "davies_bouldin": dbi}

results = []
labels_by_model = {}

if len(X_cluster) >= 3 and X_cluster.shape[1] >= 1:
    # K-means: test a small range of k values.
    for k in range(2, min(6, len(X_cluster) - 1) + 1):
        try:
            labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_cluster)
            row = evaluate_labels(X_cluster, labels, f"K-means (k={k})")
            results.append(row)
            labels_by_model[row["model"]] = labels
        except Exception as exc:
            print(f"K-means k={k} skipped:", exc)

    # Agglomerative clustering.
    try:
        labels = AgglomerativeClustering(n_clusters=2).fit_predict(X_cluster)
        row = evaluate_labels(X_cluster, labels, "Agglomerative")
        results.append(row)
        labels_by_model["Agglomerative"] = labels
    except Exception as exc:
        print("Agglomerative clustering skipped:", exc)

    # DBSCAN; parameters may need tuning for a particular dataset.
    try:
        labels = DBSCAN(eps=1.5, min_samples=5, metric="jaccard").fit_predict(X_cluster)
        row = evaluate_labels(X_cluster, labels, "DBSCAN")
        results.append(row)
        labels_by_model["DBSCAN"] = labels
    except Exception as exc:
        print("DBSCAN skipped:", exc)

    # Spectral clustering is expensive, so run on a smaller sample if necessary.
    try:
        spectral_n = min(800, len(X_cluster))
        spectral_idx = np.arange(spectral_n)
        labels = SpectralClustering(
            n_clusters=2, affinity="nearest_neighbors",
            n_neighbors=min(10, spectral_n - 1),
            random_state=RANDOM_STATE
        ).fit_predict(X_cluster[spectral_idx])
        row = evaluate_labels(X_cluster[spectral_idx], labels, "Spectral (subset)")
        results.append(row)
        labels_by_model["Spectral (subset)"] = labels
    except Exception as exc:
        print("Spectral clustering skipped:", exc)
else:
    print("Clustering skipped: at least 3 transactions are needed.")

results_df = pd.DataFrame(results)
display(results_df.sort_values("silhouette", ascending=False, na_position="last"))

## 9. View K-means cluster sizes and product profiles

The following uses the best-scoring K-means configuration from the tested range when a valid score exists. Cluster profiles describe products that are relatively common in each cluster; they are not exclusive customer types.


In [ ]:
kmeans_rows = results_df[
    results_df["model"].str.startswith("K-means", na=False) &
    results_df["silhouette"].notna()
] if not results_df.empty else pd.DataFrame()

if not kmeans_rows.empty:
    best_model_name = kmeans_rows.sort_values("silhouette", ascending=False).iloc[0]["model"]
    best_labels = labels_by_model[best_model_name]
    print("Selected model:", best_model_name)
    display(pd.Series(best_labels).value_counts().sort_index().rename_axis("cluster").to_frame("transactions"))

    # Product prevalence per cluster, using the same rows used for K-means.
    cluster_profile = []
    for cluster_id in sorted(np.unique(best_labels)):
        cluster_rows = X_cluster[best_labels == cluster_id]
        prevalence = cluster_rows.mean(axis=0)
        top_idx = np.argsort(prevalence)[::-1][:min(10, len(all_products))]
        for j in top_idx:
            cluster_profile.append({
                "cluster": int(cluster_id),
                "product": all_products[j],
                "share_of_baskets": float(prevalence[j])
            })
    cluster_profile_df = pd.DataFrame(cluster_profile)
    display(cluster_profile_df)
else:
    print("No valid K-means score was available to select a model.")

## 10. Business interpretation and limitations

Possible uses:
1. Explore products that often appear in baskets and test possible product bundles.
2. Use cluster profiles to form hypotheses for cross-selling or product placement.
3. Compare clustering with association-rule mining and basket-similarity measures such as Jaccard distance.

Limitations:
- Results depend on the quality and format of the input CSV.
- This analysis uses binary product presence/absence, not quantities, prices, or revenue.
- PCA is only a two-dimensional approximation.
- Clusters show basket similarity and do not prove causal customer behaviour.
- Any business recommendation should be tested against additional data before implementation.


## 11. Conclusion

This notebook summarizes transaction basket sizes and product frequencies, creates a binary transaction–product matrix, visualizes transactions with PCA, and compares K-means, Agglomerative Clustering, DBSCAN, and Spectral Clustering where feasible. Use the actual output tables and charts from your dataset when writing the final findings; do not report example values that were not produced by your run.
